# MiniMax H3 Turbo LoRA：Colab 首幀／多參照圖片轉影片（含音訊）

這份 notebook 使用 ComfyUI 在 Colab 遠端 GPU 執行 MiniMax H3。網頁 notebook 預設使用 FL2VA：提供**一張首幀圖**與 `[Shot N]` 分鏡，生成含同步音訊的 MP4。本機 CLI 會切換成 Ref2VA，可提供 1–9 張語意參照圖和完整 prompt 文字檔。模型會依 CUDA 與 GPU 能力選擇 INT8、FP8 或 BF16 權重。Ref2VA 圖片依上傳順序標記為 `<Picture 1>`、`<Picture 2>` 等；它們是內容參照，不會自動固定到特定影片時間。

可直接在 Colab 網頁 notebook 使用，也可從本機終端透過 `run_colab_inference.sh` 執行，不必開啟 Colab notebook 網頁。CLI 模式會把本機參照圖與 prompt.txt 上傳到遠端 VM，完成後下載 MP4；模型運算仍在 Colab GPU，不是在 Mac 本機執行。首次執行需下載大型模型與 LoRA；建議高 RAM、A100 等級 GPU 和至少 60 GiB 可用磁碟；若 Ref2VA 選用 BF16 備援權重，需至少 80 GiB。免費 T4 通常不足。模型受 [MiniMax H3 授權](https://huggingface.co/MiniMaxAI/MiniMax-H3)約束。

從上到下執行各 cell：**安裝 → 下載模型 → 設定模式與參數 → 設定圖片 → 檢查 prompt → 推論 → 儲存 MP4**。網頁模式預設使用 FL2VA，會顯示預覽並提供下載；CLI 模式使用 Ref2VA，結果存到 `/content/minimax_h3_output.mp4` 供終端下載。

兩種模式都使用官方 Qwen3-VL 文字編碼器與雙 VAE 解碼。FL2VA 模式使用本 repo 的 v4 Turbo LoRA；Ref2VA 使用官方 ref2v 4-step LoRA 與對應模型。原 workflow 的 SageAttention、SolAttn、Spectrum、預覽與除錯節點為可略過的輔助節點；輸出可能與桌面 workflow 不完全相同。

參考：[MiniMax 官方模型卡](https://huggingface.co/MiniMaxAI/MiniMax-H3)、[官方 Ref2VA prompt 指南](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/docs/VIDEO_PROMPT_WRITING_GUIDE_ref_en.md)、[FL2VA prompt 指南](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/docs/VIDEO_PROMPT_WRITING_GUIDE_base_en.md)、[ComfyUI H3 節點](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy_extras/nodes_minimax_h3.py)、[原 workflow](https://huggingface.co/drbaph/MiniMax-H3-Turbo-Lora-ComfyUI/blob/main/fl_minimax_h3_turbo_lora_example_workflow.json)。


## 生成時間：最短、最長與實際秒數

| 來源／用途 | 最短 | 最長 | 這份 notebook 的處理 |
| --- | ---: | ---: | --- |
| [MiniMax H3 模型卡](https://huggingface.co/MiniMaxAI/MiniMax-H3)的輸出規格 | 4 秒 | 15 秒 | 接受 `DURATION_SECONDS` 介於 4–15 |
| [ComfyUI H3 節點](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy_extras/nodes_minimax_h3.py)標示的約略已訓練範圍 | 約 5 秒（124 幀） | 約 15 秒（362 幀） | **建議使用 5–15 秒**；4 秒屬較不穩定的邊界設定 |

H3 以 **24 fps** 生成，ComfyUI 將要求幀數向上對齊到 `17k+5`。所以 `DURATION_SECONDS` 是**要求秒數**，不一定等於 MP4 的精確秒數。常見例子：4 → **107 幀 / 4.46 秒**；5 → **124 幀 / 5.17 秒**；12 → **294 幀 / 12.25 秒**；15 → **362 幀 / 15.08 秒**。若成品必須**嚴格不超過 15.00 秒**，在 notebook 設 `DURATION_SECONDS = 14.375`，對應 345 幀 / 14.375 秒；也可以在輸出後另行裁切，但會改變結尾音訊。

多鏡頭的切換時間要以影片**起點**為 0 秒、嚴格遞增、落在要求秒數之內。例如 12 秒影片可在 4.0 與 8.0 秒切成三個 shots。這些是 prompt 中希望模型切換的時間，不是剪輯軟體的硬切點。

ComfyUI 節點的數值欄位允許更短或更長的幀數，但那是介面範圍，**不代表模型在該範圍內經過訓練或保證品質**。


## 1. 檢查 Colab 環境並安裝 ComfyUI

本機 CLI 會自動建立遠端 GPU runtime；用 Colab 網頁時，請先在「執行階段 → 變更執行階段類型」選 GPU。下一個 cell 在 `/content` 安裝 ComfyUI 和影片輸出節點；需有 `ffmpeg`、`ffprobe` 與至少 60 GiB 可用磁碟；Ref2VA 的 BF16 備援模式需要 80 GiB。


In [ ]:
import os, shutil, subprocess, sys, torch
from pathlib import Path

assert torch.cuda.is_available(), "請在 Colab「執行階段 > 變更執行階段類型」選擇 GPU。"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("PyTorch CUDA:", torch.version.cuda)
print("Free disk GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))
# 網頁 notebook 可改為 "reference"；CLI launcher 會自動設定。
INFERENCE_MODE = os.environ.get("H3_INFERENCE_MODE", "first_frame").strip().lower()
assert INFERENCE_MODE in {"first_frame", "reference"}, "H3_INFERENCE_MODE 請使用 first_frame 或 reference。"
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
CUDA_MAJOR = int((torch.version.cuda or "0").split(".", 1)[0])
if INFERENCE_MODE == "reference":
    if CUDA_MAJOR >= 13 and GPU_CAPABILITY >= (8, 0):
        REF2VA_VARIANT = "int8_convrot"
    elif GPU_CAPABILITY >= (8, 9):
        REF2VA_VARIANT = "fp8_scaled"
    else:
        REF2VA_VARIANT = "bf16"
else:
    REF2VA_VARIANT = None
MINIMUM_FREE_DISK = 80 * 2**30 if REF2VA_VARIANT == "bf16" else 60 * 2**30
assert shutil.disk_usage("/content").free >= MINIMUM_FREE_DISK, (
    f"目前選用 {REF2VA_VARIANT or 'FL2VA'} 模型，至少需要 {MINIMUM_FREE_DISK / 2**30:.0f} GiB 可用磁碟。"
)
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "需要 ffmpeg 與 ffprobe 才能輸出並檢查 MP4 音訊。"
COMFY = Path("/content/ComfyUI")
VHS = COMFY / "custom_nodes" / "ComfyUI-VideoHelperSuite"
SETUP_MARKER = Path("/content/.h3_comfyui_setup_v1")
if SETUP_MARKER.exists():
    assert COMFY.is_dir() and VHS.is_dir(), "Colab setup marker exists, but ComfyUI files are missing; remove /content/.h3_comfyui_setup_v1 and retry."
    print("重用本次 Colab session 已安裝的 ComfyUI 環境")
else:
    if not COMFY.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt"), "huggingface_hub", "hf_xet", "pillow", "requests"], check=True)
    if not VHS.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git", str(VHS)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(VHS / "requirements.txt")], check=True)
    SETUP_MARKER.touch()
print("ComfyUI:", COMFY)


## 2. 下載模型

`INFERENCE_MODE` 在環境檢查 cell 預設為 `first_frame`；本機 CLI 會自動設為 `reference`。網頁模式可在環境檢查 cell 選擇 `reference`。Ref2VA 會依 CUDA 版本與 GPU compute capability 選用官方 INT8 ConvRot、FP8 scaled 或 BF16 diffusion 權重，並配對 video VAE 與 4-step Turbo LoRA；BF16 備援需至少 80 GiB 可用磁碟。FL2VA 仍可調 `DIFFUSION_VARIANT`。文字編碼器和 audio VAE 共用。首次執行需要下載大型模型；Colab 執行階段重建後可能需要重新下載。


In [ ]:
from huggingface_hub import hf_hub_download

BASE_REPO = "Comfy-Org/MiniMax-H3"
TEXT_ENCODER = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
AUDIO_VAE = "minimax_h3_audio_vae_fp32.safetensors"

if INFERENCE_MODE == "reference":
    DIFFUSION = f"minimax_h3_ref2va_pruned_{REF2VA_VARIANT}.safetensors"
    VIDEO_VAE = "minimax_h3_video_vae_int8_convrot.safetensors" if REF2VA_VARIANT == "int8_convrot" else "minimax_h3_video_vae_fp16.safetensors"
    LORA_REPO = BASE_REPO
    LORA_REMOTE = "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"
    print("Ref2VA variant:", REF2VA_VARIANT, "(CUDA", torch.version.cuda, "; GPU capability", GPU_CAPABILITY, ")")
else:
    LORA_REPO = "drbaph/MiniMax-H3-Turbo-Lora-ComfyUI"
    DIFFUSION_VARIANT = "auto"  # 可改為 "fp8_scaled" 或 "int8_convrot"。
    assert DIFFUSION_VARIANT in {"auto", "fp8_scaled", "int8_convrot"}
    if DIFFUSION_VARIANT == "int8_convrot" and not (torch.version.cuda or "").startswith("13."):
        raise ValueError("官方建議 int8 ConvRot 使用 CUDA 13.x PyTorch；請改用 auto 或 fp8_scaled。")
    use_int8 = DIFFUSION_VARIANT == "int8_convrot" or (
        DIFFUSION_VARIANT == "auto" and (torch.version.cuda or "").startswith("13."))
    DIFFUSION = ("minimax_h3_fl2va_pruned_int8_convrot.safetensors" if use_int8
                 else "minimax_h3_fl2va_pruned_fp8_scaled.safetensors")
    VIDEO_VAE = "minimax_h3_video_vae_fp16.safetensors"
    LORA_REMOTE = "minimax_h3_turbo_v4_step600_ema_pruned_comfyui.safetensors"

model_files = [
    (BASE_REPO, "diffusion_models/" + DIFFUSION, "diffusion_models"),
    (BASE_REPO, "text_encoders/" + TEXT_ENCODER, "text_encoders"),
    (BASE_REPO, "vae/" + VIDEO_VAE, "vae"),
    (BASE_REPO, "vae/" + AUDIO_VAE, "vae"),
    (LORA_REPO, LORA_REMOTE, "loras"),
]
LORA = Path(LORA_REMOTE).name
for repo, remote_name, folder in model_files:
    target = COMFY / "models" / folder / Path(remote_name).name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.is_file() or target.stat().st_size == 0:
        local_dir = COMFY / "models" if repo == BASE_REPO else COMFY / "models" / "loras"
        downloaded = Path(hf_hub_download(repo_id=repo, filename=remote_name, local_dir=local_dir))
        assert downloaded == target, f"下載路徑不符：{downloaded} != {target}"
    print(folder, target.name, f"{target.stat().st_size / 2**30:.2f} GiB")


## 3. 設定多個 shots 與可調參數

先修改下一個程式 cell。`SHOTS` 清單就是分鏡表：每一項的 `start_seconds` 是**鏡頭開始時間**，`description` 寫畫面、角色動作、鏡頭運動，以及該 shot 中出現的對白或可聽見的動作聲。第一個 shot 固定 0 秒；後續 shot 會自動寫成官方格式：`[Shot 2] At 00:04.000, the camera cuts to ...`。要新增鏡頭，複製清單內一筆並修改開始時間及描述；要只有一鏡，保留第一筆即可。

**prompt 怎麼寫：**Shot 1 先描述上傳圖片 `<Picture 1>` 中既有的人物、場景、衣物與構圖，再寫動作。後續 shot 延續相同角色，明確說明新景別或新視角、切鏡後的動作。鏡頭運動可用「push in / pan right / tracking shot」加上幅度與速度。若有人說話，把對白放在該 shot，例如 `The woman (S1) says: <d>[Chinese] 你好。</d>`；同一人跨 shot 保持相同 `(S1)`。`OVERALL_SOUNDSCAPE` 寫全片環境與動作聲；`NON_DIEGETIC_MUSIC` 寫觀眾能聽、場內人物聽不到的配樂，沒有就填 `N/A`。對白不要再抄進 soundscape。官方提示詞結構見[寫作指南](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/docs/VIDEO_PROMPT_WRITING_GUIDE_base_en.md)。

| 設定 | 預設 | 用途與建議 |
| --- | --- | --- |
| `WIDTH`, `HEIGHT` | 1376 × 768 | 原 workflow 16:9、約 1 MP；兩者需為 32 的倍數。降低尺寸可減少記憶體與時間，但畫質會下降。H3 原生通常以短邊 768px 為目標。 |
| `DURATION_SECONDS` | 12 | 整支影片的要求長度，允許 4–15 秒，建議 5–15 秒；對齊後實際秒數見上一節。 |
| `SHOTS` | 三鏡 | 同一次生成中的分鏡與切鏡時間；不是三次模型推論。模型對切鏡的遵循屬生成行為，不能保證逐幀精準。 |
| `OVERALL_SOUNDSCAPE`, `NON_DIEGETIC_MUSIC` | 雨林環境聲、無配樂 | 分別控制環境／動作聲與觀眾聽到的配樂；在 shots 的描述中寫對白與場景內音樂。 |
| `SEED` | 原 workflow 固定值 | 改變起始噪聲；相同 seed 與參數有助比較，但不保證跨版本得到完全相同影片。 |
| `STEPS` | 4（Ref2VA）／8（FL2VA） | Ref2VA 官方 Turbo LoRA 使用 4 步；FL2VA 的 v4 Turbo LoRA 原 workflow 使用 6–8 步。增加步數會更慢，效果不一定更好。 |
| `LORA_STRENGTH` | 1.0 | LoRA 影響程度；可先試 0.8–1.2。0 表示不用 LoRA，可能需要較多採樣步數。 |
| `SAMPLER_NAME`, `SCHEDULER_NAME` | `euler`, `beta` | 原 workflow 的取樣器與時間排程；改動可能同時影響影片與音訊品質。 |
| `DENOISE` | 1.0 | 對空白 latent 做完整生成；此流程建議維持 1.0。 |
| `VIDEO_CRF` | 20 | MP4 壓縮品質；數字越低通常檔案越大、畫面越清楚，H.264 可用 0–51。只影響輸出壓縮，不影響模型生成。 |
| `OUTPUT_PREFIX` | `MiniMax_H3_Colab` | 輸出檔名前綴。 |
| `FIRST_FRAME_MEGAPIXELS`, `FIRST_FRAME_RESOLUTION_STEPS`, `FIRST_FRAME_UPSCALE_METHOD` | `1.0`, `32`, `nearest-exact` | 僅 FL2VA 使用：送入 H3 前縮放首幀；Ref2VA 使用 `REFERENCE_IMAGE_SIZE` 另行處理參照圖。 |
| `LOW_VRAM_MODE` | `auto` | `auto` 在 GPU VRAM 少於 48 GiB 時啟用 ComfyUI `--lowvram`；`on` 強制啟用，`off` 關閉。省 VRAM 通常會更慢；更改後請重新啟動 Colab 執行階段，再由上往下執行。 |
| `PROMPT_OVERRIDE` | 空字串 | 填入完整 prompt 時直接使用。Ref2VA 模式需使用官方參照格式；CLI 會優先讀取 `--prompt` 指定的 UTF-8 文字檔。 |

**提示詞組裝的界線：**FL2VA 會把 shot 與聲音欄位排成官方格式，**不會自動改寫或潤飾內容**；Ref2VA 直接使用完整 prompt。官方 H3-Context-IR 是另一套未隨開源權重提供的前處理服務。

**圖片模式：**`first_frame` 使用 FL2VA 與單張首幀；`reference` 使用 Ref2VA，接受最多 9 張語意參照圖片，依序對應 `<Picture 1>` 至 `<Picture 9>`。參照圖不會自動成為不同 shot 的時間軸 keyframe。H3 以 24 fps 輸出；影片與音訊各用自己的 VAE。


In [ ]:
import math

# 只改這個 cell 的設定，再從這裡往下執行。
WIDTH, HEIGHT = 1376, 768
DURATION_SECONDS = float(os.environ.get("H3_DURATION_SECONDS", "12.0"))
STEPS = 4 if INFERENCE_MODE == "reference" else 8
SEED = int(os.environ.get("H3_SEED", "768914106098754"))
LORA_STRENGTH = 1.0
SAMPLER_NAME = "euler"
SCHEDULER_NAME = "beta"
DENOISE = 1.0
VIDEO_CRF = 20
OUTPUT_PREFIX = os.environ.get("H3_OUTPUT_PREFIX", "MiniMax_H3_Colab")
REFERENCE_IMAGE_SIZE = "match"  # "match" 較快；"max" 可保留較多身份細節但較慢。
FIRST_FRAME_MEGAPIXELS = 1.0
FIRST_FRAME_RESOLUTION_STEPS = 32
FIRST_FRAME_UPSCALE_METHOD = "nearest-exact"
LOW_VRAM_MODE = "auto"  # "auto"、"on" 或 "off"

# 一個清單項目 = 同一支影片內的一個 shot。起點以秒為單位，從影片開頭算起。
SHOTS = [
    {
        "start_seconds": 0.0,
        "description": (
            "Photorealistic wildlife documentary. The chameleon in <Picture 1> "
            "remains on the same wet moss-covered branch, preserving its shape, "
            "large turreted eye, textured scales and rainforest background. "
            "A medium macro shot follows it as it starts walking forward slowly, "
            "placing one foot at a time while water droplets tremble on its skin."
        ),
    },
    {
        "start_seconds": 4.0,
        "description": (
            "a close-up of the same chameleon's face. "
            "Its eye rotates independently while its skin gradually shifts from "
            "green to turquoise and yellow. The camera pushes in with small "
            "amplitude at slow speed; the wet scales remain sharply detailed."
        ),
    },
    {
        "start_seconds": 8.0,
        "description": (
            "a wider tracking view of the same branch. "
            "The chameleon continues forward without changing identity or "
            "location, with orange and red accents passing smoothly across its "
            "scales before returning to multicoloured green. It ends still "
            "walking through soft rainforest daylight."
        ),
    },
]
OVERALL_SOUNDSCAPE = (
    "Quiet rainforest ambience with faint birds, soft insects, dripping water "
    "and leaf rustling. Subtle wet contact sounds follow the chameleon's feet "
    "as they grip and release the mossy branch."
)
NON_DIEGETIC_MUSIC = "N/A"
PROMPT_OVERRIDE = ""  # 填入完整 prompt 才會覆蓋 shot 組裝；CLI 會改讀 --prompt 檔案。

assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0, "寬高必須為正整數且是 32 的倍數。"
assert isinstance(DURATION_SECONDS, (int, float)) and math.isfinite(DURATION_SECONDS) and 4 <= DURATION_SECONDS <= 15, "請將影片長度設為 4–15 秒；5–15 秒較穩定。"
assert isinstance(STEPS, int) and STEPS > 0, "STEPS 必須是正整數。"
assert isinstance(SEED, int) and 0 <= SEED < 2**64, "SEED 必須是 0 到 2**64-1 的整數。"
assert math.isfinite(LORA_STRENGTH) and 0 <= LORA_STRENGTH <= 2, "LoRA 強度請設為 0–2。"
assert 0 < DENOISE <= 1, "DENOISE 請設為 0–1 之間（不含 0）。"
assert isinstance(VIDEO_CRF, int) and 0 <= VIDEO_CRF <= 51, "VIDEO_CRF 請設為 0–51 的整數。"
assert isinstance(OUTPUT_PREFIX, str) and OUTPUT_PREFIX.strip(), "請設定輸出檔名前綴。"
assert REFERENCE_IMAGE_SIZE in {"match", "max"}, "REFERENCE_IMAGE_SIZE 請使用 match 或 max。"
assert math.isfinite(FIRST_FRAME_MEGAPIXELS) and 0.01 <= FIRST_FRAME_MEGAPIXELS <= 16, "首幀縮放像素數請設為 0.01–16 MP。"
assert isinstance(FIRST_FRAME_RESOLUTION_STEPS, int) and FIRST_FRAME_RESOLUTION_STEPS > 0, "首幀尺寸倍數必須是正整數。"
assert LOW_VRAM_MODE in {"auto", "on", "off"}, "LOW_VRAM_MODE 請使用 auto、on 或 off。"

requested_frames = max(5, round(DURATION_SECONDS * 24))
FRAMES = requested_frames + (5 - requested_frames % 17) % 17
ACTUAL_SECONDS = FRAMES / 24
print(f"模式：{INFERENCE_MODE}；要求長度：{DURATION_SECONDS:.3f} 秒；排程：{FRAMES} 幀 / {ACTUAL_SECONDS:.3f} 秒（24 fps）")
if FRAMES < 124:
    print("提醒：此設定低於 ComfyUI H3 節點標示的約 124 幀已訓練下限，畫面/音訊可能不穩定。")
if ACTUAL_SECONDS > 15:
    print("提醒：對齊後稍長於 15 秒；若需嚴格不超過 15 秒，請設 14.375 秒或輸出後裁切。")


### 3.1 組裝並預覽模型真正收到的 prompt

FL2VA 模式會依照官方首幀格式組裝 `SHOTS`、環境聲與配樂。Ref2VA 模式則直接使用完整自訂 prompt；CLI 會從本機 `--prompt` 檔案讀取原文，讓 `<Picture N>` 對應參照圖上傳順序。兩種模式都會印出**完整的 `PROMPT`**，讓你提交前確認內容。`SHOTS` 的時間需嚴格遞增，且每個時間都小於 `DURATION_SECONDS`；每個描述都必須有實際內容。


In [ ]:
import hashlib


def shot_timestamp(seconds):
    milliseconds = round(seconds * 1000)
    minutes, remainder = divmod(milliseconds, 60_000)
    secs, millis = divmod(remainder, 1_000)
    return f"{minutes:02d}:{secs:02d}.{millis:03d}"

assert isinstance(PROMPT_OVERRIDE, str), "PROMPT_OVERRIDE 必須是字串。"
prompt_file_env = os.environ.get("H3_PROMPT_FILE")
if prompt_file_env:
    prompt_path = Path(prompt_file_env)
    assert prompt_path.is_file(), f"Colab CLI 上傳的 prompt 檔不存在：{prompt_path}"
    PROMPT = prompt_path.read_text(encoding="utf-8-sig")
    assert PROMPT.strip(), f"prompt 檔是空的：{prompt_path}"
    print("使用 CLI 上傳的完整 prompt：", prompt_path)
elif PROMPT_OVERRIDE.strip():
    PROMPT = PROMPT_OVERRIDE.strip()
    print("使用完整自訂 prompt；SHOTS、環境聲與配樂設定不參與組裝。")
elif INFERENCE_MODE == "reference":
    raise ValueError("Ref2VA 需要完整參照格式 prompt；請設定 PROMPT_OVERRIDE 或使用 CLI --prompt 檔案。")
else:
    assert isinstance(SHOTS, list) and SHOTS, "SHOTS 至少需要一個鏡頭。"
    starts = []
    cut_milliseconds = []
    parts = []
    for number, shot in enumerate(SHOTS, start=1):
        assert isinstance(shot, dict), f"Shot {number} 必須是 dict。"
        start = shot.get("start_seconds")
        description = shot.get("description")
        assert isinstance(start, (int, float)) and math.isfinite(start), f"Shot {number} 的開始時間必須是數字。"
        assert isinstance(description, str) and description.strip(), f"Shot {number} 的 description 不能空白。"
        assert 0 <= start < DURATION_SECONDS, f"Shot {number} 的開始時間必須介於 0 與要求秒數之間。"
        assert not starts or start > starts[-1], "各 shot 的開始時間必須嚴格遞增。"
        stamp_ms = round(start * 1000)
        assert not cut_milliseconds or stamp_ms > cut_milliseconds[-1], "切鏡時間以毫秒顯示，請至少間隔 0.001 秒。"
        starts.append(start)
        cut_milliseconds.append(stamp_ms)
        description = description.strip()
        if number == 1:
            assert start == 0, "Shot 1 必須從 0 秒開始。"
            parts.append(f"[Shot 1] {description}")
        else:
            cut_words = ("the camera cuts to", "the shot cuts to", "the shot transitions to",
                         "the shot changes to", "the shot switches to")
            transition = description if description.lower().startswith(cut_words) else "the camera cuts to " + description
            parts.append(f"[Shot {number}] At {shot_timestamp(start)}, {transition}")

    assert isinstance(OVERALL_SOUNDSCAPE, str) and OVERALL_SOUNDSCAPE.strip(), "請填寫 OVERALL_SOUNDSCAPE。"
    assert isinstance(NON_DIEGETIC_MUSIC, str) and NON_DIEGETIC_MUSIC.strip(), "無配樂時請填 N/A。"
    PROMPT = (
        "For the target video, at 0.00 seconds into the target video, "
        "<Picture 1> (from [Shot 1]) is fully referenced.\n\n"
        "integrated_multimodal_description: " + "\n".join(parts) + "\n\n"
        "overall_soundscape: " + OVERALL_SOUNDSCAPE.strip() + "\n\n"
        "non_diegetic_music: " + NON_DIEGETIC_MUSIC.strip()
    )
    print(f"已建立 {len(SHOTS)} 個 shot；要求的分鏡時間（切鏡由模型生成，非硬剪輯）：")
    for number, start in enumerate(starts, start=1):
        end = starts[number] if number < len(starts) else DURATION_SECONDS
        print(f"  Shot {number}: {start:.3f}–{end:.3f} 秒（約 {end - start:.3f} 秒）")

if os.environ.get("H3_VERBOSE_PROMPT", "").strip().lower() in {"1", "true", "yes"}:
    print("\n模型收到的完整 prompt：\n")
    print(PROMPT)
else:
    digest = hashlib.sha256(PROMPT.encode("utf-8")).hexdigest()[:12]
    print(f"\nprompt 已準備：{len(PROMPT)} 字元 / {len(PROMPT.encode('utf-8'))} bytes，sha256:{digest}")
    print("（prompt 可能含敏感內容，預設不印全文；設定 H3_VERBOSE_PROMPT=1 可在日誌中顯示。）")


### 3.2 設定圖片參照

FL2VA 模式請提供一張首幀圖片；Ref2VA 模式可提供 1–9 張 PNG、JPEG、WebP 等圖片。Ref2VA 會依輸入順序將它們標記為 `<Picture 1>`、`<Picture 2>` 等，供完整 prompt 描述如何使用。語意參照圖片不會自動固定在影片的不同時間點。

Colab 網頁模式會在下一個 cell 顯示上傳按鈕。本機 CLI 會把每個 `--image` 路徑上傳到遠端，並以 `H3_REFERENCE_IMAGES` 傳入 notebook；CLI 也會上傳 `--prompt` 文字檔，cell 會直接讀取其 UTF-8 內容。


In [ ]:
import json
from io import BytesIO
from PIL import Image

CLI_MODE = False
REFERENCE_IMAGE_NAMES = []
REFERENCE_IMAGE_PATHS = []

if INFERENCE_MODE == "reference":
    configured_references = os.environ.get("H3_REFERENCE_IMAGES")
    if configured_references:
        source_paths = json.loads(configured_references)
        assert isinstance(source_paths, list) and 1 <= len(source_paths) <= 9, "Ref2VA 請提供 1–9 張參照圖。"
        source_images = []
        for source_path in source_paths:
            path = Path(source_path)
            assert path.is_file(), f"參照圖片不存在：{path}"
            source_images.append((path.name, path))
        CLI_MODE = True
    else:
        from google.colab import files
        uploaded = files.upload()
        assert 1 <= len(uploaded) <= 9, "Ref2VA 請上傳 1–9 張參照圖片。"
        source_images = [(name, BytesIO(image_bytes)) for name, image_bytes in uploaded.items()]

    input_dir = COMFY / "input"
    input_dir.mkdir(parents=True, exist_ok=True)
    for number, (source_name, image_source) in enumerate(source_images, start=1):
        with Image.open(image_source) as opened:
            image = opened.convert("RGB")
        image_path = input_dir / f"h3_reference_{number:02d}.png"
        image.save(image_path)
        REFERENCE_IMAGE_NAMES.append(image_path.name)
        REFERENCE_IMAGE_PATHS.append(image_path)
        print(f"<Picture {number}>：{source_name} → {image_path} ({image.size[0]}×{image.size[1]})")
else:
    configured_image = os.environ.get("H3_INPUT_IMAGE")
    source_path = Path(configured_image or "/content/h3_first_frame.png")
    CLI_MODE = source_path.is_file()
    assert not configured_image or CLI_MODE, f"Colab CLI 上傳的首幀不存在：{source_path}"
    if CLI_MODE:
        with Image.open(source_path) as opened:
            image = opened.convert("RGB")
        print("使用 CLI 上傳的首幀：", source_path)
    else:
        from google.colab import files
        uploaded = files.upload()
        assert len(uploaded) == 1, "FL2VA 請只上傳一張首幀圖片。"
        name, image_bytes = next(iter(uploaded.items()))
        with Image.open(BytesIO(image_bytes)) as opened:
            image = opened.convert("RGB")
        print("使用網頁上傳的首幀：", name)

    if abs(image.width / image.height - WIDTH / HEIGHT) > 0.01:
        print("提醒：原圖比例與輸出不同，首幀會被拉伸到影片畫布。")
    input_path = COMFY / "input" / "h3_colab_first_frame.png"
    input_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(input_path)
    print("ComfyUI 首幀：", input_path, image.size)


## 4. 啟動 ComfyUI 並提交推論

ComfyUI 只在 Colab 的 `127.0.0.1:8188` 執行。第一個 cell 啟動服務並依模式確認節點；第二個 cell 把設定、圖片參照與 `PROMPT` 送入 ComfyUI。若模型或節點名稱不符，會顯示具體錯誤。每次修改第 3 節設定後，請重新執行**設定 → prompt 組裝 → 提交 → 等待輸出**；更換圖片時也要重新執行圖片參照 cell。


In [ ]:
import json, time, requests

PORT = 8188
BASE_URL = f"http://127.0.0.1:{PORT}"
LOG_PATH = Path("/content/comfyui_h3.log")
try:
    response = requests.get(BASE_URL + "/system_stats", timeout=2)
    response.raise_for_status()
    print("使用已啟動的 ComfyUI")
except requests.RequestException:
    LOG_FILE = open(LOG_PATH, "a")
    launch = [sys.executable, str(COMFY / "main.py"), "--listen", "127.0.0.1", "--port", str(PORT), "--disable-auto-launch"]
    if LOW_VRAM_MODE == "on" or (LOW_VRAM_MODE == "auto" and torch.cuda.get_device_properties(0).total_memory < 48 * 2**30):
        launch.append("--lowvram")
    COMFY_PROCESS = subprocess.Popen(launch, cwd=COMFY, stdout=LOG_FILE, stderr=subprocess.STDOUT)
    for _ in range(180):
        if COMFY_PROCESS.poll() is not None:
            raise RuntimeError(f"ComfyUI 啟動失敗；查看 {LOG_PATH}\n" + LOG_PATH.read_text()[-5000:])
        try:
            requests.get(BASE_URL + "/system_stats", timeout=2).raise_for_status()
            break
        except requests.RequestException:
            time.sleep(2)
    else:
        raise TimeoutError(f"ComfyUI 啟動逾時；查看 {LOG_PATH}")

object_info = requests.get(BASE_URL + "/object_info", timeout=30).json()
required_nodes = {"UNETLoader", "CLIPLoader", "VAELoader", "LoraLoaderModelOnly",
                  "LoadImage", "BasicGuider", "RandomNoise", "KSamplerSelect",
                  "BasicScheduler", "SamplerCustomAdvanced", "VAEDecode", "VAEDecodeAudio",
                  "VHS_VideoCombine"}
if INFERENCE_MODE == "reference":
    required_nodes.add("MiniMaxH3ReferenceToVideo")
else:
    required_nodes.update({"ImageScaleToTotalPixels", "MiniMaxH3ImageToVideo"})
missing = required_nodes - object_info.keys()
assert not missing, f"ComfyUI 缺少節點：{sorted(missing)}。請查看 {LOG_PATH}，並確認 ComfyUI / VideoHelperSuite 安裝成功。"
print("節點檢查通過，ComfyUI 已啟動")


In [ ]:
import re

graph = {
    "1": {"class_type": "UNETLoader", "inputs": {"unet_name": DIFFUSION, "weight_dtype": "default"}},
    "2": {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": LORA, "strength_model": LORA_STRENGTH}},
    "3": {"class_type": "CLIPLoader", "inputs": {"clip_name": TEXT_ENCODER, "type": "minimax", "device": "default"}},
    "4": {"class_type": "VAELoader", "inputs": {"vae_name": VIDEO_VAE}},
    "5": {"class_type": "VAELoader", "inputs": {"vae_name": AUDIO_VAE}},
}

if INFERENCE_MODE == "reference":
    picture_inputs = {
        "clip": ["3", 0], "vae": ["4", 0], "audio_vae": ["5", 0],
        "prompt": PROMPT, "width": WIDTH, "height": HEIGHT, "length": FRAMES,
        "ref_image_size": REFERENCE_IMAGE_SIZE,
    }
    for index, image_name in enumerate(REFERENCE_IMAGE_NAMES):
        node_id = str(100 + index)
        graph[node_id] = {"class_type": "LoadImage", "inputs": {"image": image_name}}
        picture_inputs[f"ref_images.ref_image_{index}"] = [node_id, 0]
    picture_count = len(REFERENCE_IMAGE_NAMES)
    referenced_pictures = [int(match.group(1)) for match in re.finditer(r"<Picture\s+(\d+)>", PROMPT)]
    invalid_pictures = sorted({number for number in referenced_pictures if number < 1 or number > picture_count})
    if invalid_pictures:
        raise ValueError(f"prompt 使用了不存在的圖片標籤：{invalid_pictures}；目前提供 {picture_count} 張，標籤為 <Picture 1>–<Picture {picture_count}>。")
    graph["7"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": picture_inputs}
else:
    graph.update({
        "6": {"class_type": "LoadImage", "inputs": {"image": input_path.name}},
        "16": {"class_type": "ImageScaleToTotalPixels", "inputs": {
            "image": ["6", 0], "upscale_method": FIRST_FRAME_UPSCALE_METHOD,
            "megapixels": FIRST_FRAME_MEGAPIXELS, "resolution_steps": FIRST_FRAME_RESOLUTION_STEPS}},
        "7": {"class_type": "MiniMaxH3ImageToVideo", "inputs": {
            "clip": ["3", 0], "vae": ["4", 0], "first_frame": ["16", 0],
            "prompt": PROMPT, "width": WIDTH, "height": HEIGHT, "length": FRAMES}},
    })

graph.update({
    "8": {"class_type": "BasicGuider", "inputs": {"model": ["2", 0], "conditioning": ["7", 0]}},
    "9": {"class_type": "RandomNoise", "inputs": {"noise_seed": SEED}},
    "10": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": SAMPLER_NAME}},
    "11": {"class_type": "BasicScheduler", "inputs": {"model": ["2", 0], "scheduler": SCHEDULER_NAME, "steps": STEPS, "denoise": DENOISE}},
    "12": {"class_type": "SamplerCustomAdvanced", "inputs": {
        "noise": ["9", 0], "guider": ["8", 0], "sampler": ["10", 0],
        "sigmas": ["11", 0], "latent_image": ["7", 1]}},
    "13": {"class_type": "VAEDecode", "inputs": {"samples": ["12", 0], "vae": ["4", 0]}},
    "14": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["12", 0], "vae": ["5", 0]}},
    "15": {"class_type": "VHS_VideoCombine", "inputs": {
        "images": ["13", 0], "audio": ["14", 0], "frame_rate": 24,
        "loop_count": 0, "filename_prefix": OUTPUT_PREFIX,
        "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": VIDEO_CRF,
        "save_metadata": True, "trim_to_audio": False,
        "pingpong": False, "save_output": True}},
})

# Ref2VA reference slots come from an autogrow group, so /object_info advertises
# them either grouped ("ref_images") or expanded ("ref_images.ref_image_0").
# Accept only what the running ComfyUI actually advertises: a dropped reference
# image is the worst failure mode here, because the render still "succeeds".
reference_schema = object_info["MiniMaxH3ReferenceToVideo"]["input"] if INFERENCE_MODE == "reference" else {}
reference_known = set(reference_schema.get("required", {})) | set(reference_schema.get("optional", {}))
advertised_reference_keys = {key for key in reference_known if re.fullmatch(r"ref_images\.ref_image_\d+", key)}
used_reference_keys = {f"ref_images.ref_image_{index}" for index in range(len(REFERENCE_IMAGE_NAMES))} if INFERENCE_MODE == "reference" else set()
if INFERENCE_MODE == "reference":
    if advertised_reference_keys:
        missing_slots = sorted(used_reference_keys - advertised_reference_keys)
        if missing_slots:
            raise ValueError(f"ComfyUI 的 MiniMaxH3ReferenceToVideo 不再提供這些參照槽位：{missing_slots}；請更新 notebook。")
        allowed_reference_keys = used_reference_keys
    elif "ref_images" in reference_known:
        allowed_reference_keys = used_reference_keys
    else:
        raise ValueError("ComfyUI 的 MiniMaxH3ReferenceToVideo 沒有宣告 ref_images 輸入；ref2va 參照圖會被安靜忽略，請更新 notebook。")
else:
    allowed_reference_keys = set()

for node in graph.values():
    spec = object_info[node["class_type"]]["input"]
    known = set(spec.get("required", {})) | set(spec.get("optional", {}))
    if node["class_type"] == "VHS_VideoCombine":
        format_info = spec["required"]["format"][1]["formats"]
        format_name = node["inputs"]["format"]
        if format_name not in format_info:
            raise ValueError(f"VideoHelperSuite 不支援輸出格式：{format_name}")
        known.update(widget[0] for widget in format_info[format_name])
    unexpected = set(node["inputs"]) - known - allowed_reference_keys
    if unexpected:
        raise ValueError(f"{node['class_type']} 的參數已變更：{sorted(unexpected)}；請更新 notebook。")
result = requests.post(BASE_URL + "/prompt", json={"prompt": graph, "client_id": "h3-colab"}, timeout=60)
if not result.ok:
    raise RuntimeError(f"提交失敗 HTTP {result.status_code}: {result.text}")
job = result.json()
if job.get("node_errors"):
    raise RuntimeError("節點驗證失敗：" + json.dumps(job["node_errors"], ensure_ascii=False, indent=2))
prompt_id = job["prompt_id"]
print("已提交推論：", prompt_id)


## 5. 等待、檢查並儲存 MP4

大型模型首次載入可能需要數分鐘或更久。輸出 cell 最多等待兩小時，檢查 MP4 同時有影片與音訊串流。網頁模式會顯示預覽並提供下載；本機 CLI 模式會複製到 `/content/minimax_h3_output.mp4`，讓 launcher 下載到本機。音訊串流存在不代表音訊內容一定理想；若畫面或聲音不如預期，先用相同 seed 比較 prompt、時長、步數與模型版本，並查看 `/content/comfyui_h3.log`。


In [ ]:
job_timeout_seconds = float(os.environ.get("H3_JOB_TIMEOUT_SECONDS", "7200"))
assert math.isfinite(job_timeout_seconds) and job_timeout_seconds > 0, "H3_JOB_TIMEOUT_SECONDS 必須是正數。"
# A long render outlives transient proxy blips; losing the poll must not lose
# the job, because the same prompt_id is still retrievable.
MAX_POLL_FAILURES = 20
poll_failures = 0
deadline = time.monotonic() + job_timeout_seconds
last_report = 0
while time.monotonic() < deadline:
    if 'COMFY_PROCESS' in globals() and COMFY_PROCESS.poll() is not None:
        raise RuntimeError(f"ComfyUI 已退出；查看 {LOG_PATH}\n" + LOG_PATH.read_text()[-5000:])
    try:
        history = requests.get(BASE_URL + f"/history/{prompt_id}", timeout=30).json()
        poll_failures = 0
    except (requests.RequestException, ValueError) as poll_error:
        poll_failures += 1
        if poll_failures >= MAX_POLL_FAILURES:
            raise RuntimeError(
                f"連續 {poll_failures} 次無法取得推論狀態；ComfyUI 可能仍在執行，"
                f"可稍後查詢 /history/{prompt_id}。最後錯誤：{poll_error}\n日誌：{LOG_PATH}"
            ) from poll_error
        print(f"取得推論狀態失敗（{poll_failures}/{MAX_POLL_FAILURES}），繼續等待：{poll_error}")
        time.sleep(min(60, 5 * poll_failures))
        continue
    if prompt_id in history:
        run = history[prompt_id]
        if run.get("status", {}).get("status_str") == "error":
            raise RuntimeError("推論失敗：" + json.dumps(run.get("status", {}), ensure_ascii=False, indent=2) + f"\n日誌：{LOG_PATH}")
        outputs = run.get("outputs", {})
        if "15" in outputs:
            break
        if run.get("status", {}).get("status_str") == "success":
            raise RuntimeError("任務已完成但沒有 MP4 輸出：" + json.dumps(outputs, ensure_ascii=False))
    if time.monotonic() - last_report >= 60:
        print("仍在生成影片…")
        last_report = time.monotonic()
    time.sleep(10)
else:
    raise TimeoutError(f"等待推論超過 {job_timeout_seconds:g} 秒；可查詢 /history/{prompt_id}，日誌：{LOG_PATH}")

filenames = outputs["15"].get("gifs", [])
if not filenames:
    raise RuntimeError("VideoHelperSuite 未回傳 MP4 輸出；請查看 " + str(LOG_PATH))
video_info = next((item for item in filenames if item["filename"].endswith(".mp4")), None)
if video_info is None:
    raise RuntimeError("找不到 MP4；輸出：" + str(filenames))
video_path = COMFY / "output" / video_info.get("subfolder", "") / video_info["filename"]
assert video_path.is_file() and video_path.stat().st_size > 0, f"輸出檔不存在：{video_path}"

# Publish the render before inspecting it. A silent MP4 is still a usable,
# already paid-for result; failing before the copy used to throw it away.
if CLI_MODE:
    CLI_OUTPUT_PATH = Path(os.environ.get("H3_OUTPUT_PATH", "/content/minimax_h3_output.mp4"))
    CLI_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(video_path, CLI_OUTPUT_PATH)
    print("COLAB_CLI_OUTPUT_PATH:", CLI_OUTPUT_PATH)
else:
    from IPython.display import Video, display
    from google.colab import files

    display(Video(str(video_path), embed=True))
    files.download(str(video_path))

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "stream=codec_type",
     "-of", "json", str(video_path)], capture_output=True, text=True, check=True)
streams = {item["codec_type"] for item in json.loads(probe.stdout).get("streams", [])}
if {"video", "audio"} <= streams:
    print("完成：", video_path, f"({video_path.stat().st_size / 2**20:.1f} MiB，含影片與音訊)")
else:
    print("警告：MP4 已產出，但缺少預期的音訊串流：", sorted(streams))


## 常見調整與限制

- **想讓切鏡更明顯：**在新 shot 描述中寫清楚「the camera cuts to」後的新景別、視角和主體狀態。只改一點點角度時，改寫成同一 shot 的鏡頭運動通常較自然。切鏡時間由模型理解，不能保證影格精準。
- **角色在不同 shot 變樣：**每一鏡重申同一角色與場景的關鍵特徵；Shot 1 的首幀只能錨定開頭，後面仍靠文字與模型維持一致。
- **GPU 記憶體不足：**降低 `WIDTH`、`HEIGHT` 或 `DURATION_SECONDS`，再從第 3 節重跑；保持寬高都是 32 的倍數。也可先使用 5 秒、一到兩鏡確認流程。
- **音訊失真或沉默：**檢查環境聲/對白描述、取樣設定與日誌。模型同時生成影片和音訊，MP4 串流檢查只能確認音訊軌存在。
- **想提供多張語意參照圖片：**切換到 `reference` 模式；CLI launcher 會自動切換，最多可上傳 9 張，並在完整 prompt 中以 `<Picture N>` 標註。這些圖片提供語意條件，不會自動固定在不同影片時間；需要時間軸 keyframe 時，還需接上官方 `Add Guide for MiniMax H3` 節點。
- **想讀本機 prompt.txt：**使用 CLI `--prompt` 指定 UTF-8 文字檔；內容會直接成為 Ref2VA prompt，不會由 `SHOTS` 重新組裝。
- **與原 workflow 畫面不同：**本 notebook 不裝原 workflow 的可選注意力補丁、Spectrum、預覽和除錯節點，並使用官方文字編碼器。完整原流程與本版的輸出可能不同。
